# Stanford RNA 3D Folding Part 2 - Complete Notebook

Self-contained notebook for training and generating 3D structure predictions for RNA sequences.

**Competition**: [Stanford RNA 3D Folding Part 2](https://www.kaggle.com/competitions/stanford-rna-3d-folding-2)

**Task**: Predict 3D C1' atom coordinates for RNA molecules from sequence.

**Metric**: TM-score (best of 5 predictions per target)

In [ ]:
import os
import sys
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler
from einops import rearrange
from tqdm import tqdm

# BioPython for sequence alignment (TBM pipeline)
try:
    from Bio.Align import PairwiseAligner
    HAS_BIOPYTHON = True
except ImportError:
    HAS_BIOPYTHON = False
    print('WARNING: BioPython not available. TBM pipeline disabled.')

# Check environment
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Configuration

In [ ]:
# Auto-detect environment: Kaggle vs local
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    COMP_DIR = '/kaggle/input/stanford-rna-3d-folding-2'
    MODEL_DIR = '/kaggle/input/rna-fold-model'
    OUTPUT_DIR = '/kaggle/working'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
    COMP_DIR = os.path.join(PROJECT_ROOT, 'data')
    MODEL_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
    OUTPUT_DIR = PROJECT_ROOT

# Training save_dir must be writable — on Kaggle, /kaggle/input is read-only
SAVE_DIR = '/kaggle/working/checkpoints' if IS_KAGGLE else MODEL_DIR

# Number of predictions per target (competition allows up to 10)
N_PREDICTIONS = 10

CONFIG = {
    'data': {
        'train_sequences_csv': os.path.join(COMP_DIR, 'train_sequences.csv'),
        'train_labels_csv': os.path.join(COMP_DIR, 'train_labels.csv'),
        'val_sequences_csv': os.path.join(COMP_DIR, 'validation_sequences.csv'),
        'val_labels_csv': os.path.join(COMP_DIR, 'validation_labels.csv'),
        'test_csv': os.path.join(COMP_DIR, 'test_sequences.csv'),
        'max_seq_len': 512,
        'num_workers': 4,
    },
    'model': {
        'd_model': 256,
        'n_heads': 8,
        'n_layers': 8,
        'd_ff': 1024,
        'dropout': 0.1,
        'num_predictions': 5,
        'n_recycles': 1,
    },
    'training': {
        'batch_size': 4,
        'learning_rate': 3.0e-4,
        'weight_decay': 1.0e-4,
        'num_epochs': 100,
        'warmup_steps': 1000,
        'grad_clip': 1.0,
        'seed': 42,
        'save_dir': SAVE_DIR,
    },
    'tbm': {
        'min_similarity': 0.0,
        'min_percent_identity': 50.0,
        'shortlist_size': 128,
        'top_templates': 30,
        'chunk_overlap': 128,
    },
}

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Competition dir: {COMP_DIR}')
print(f'Model dir (weights): {MODEL_DIR}')
print(f'Save dir (training): {SAVE_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'N_PREDICTIONS: {N_PREDICTIONS}')

# Show available data files
if os.path.exists(COMP_DIR):
    print(f'\nFiles in {COMP_DIR}:')
    for f in sorted(os.listdir(COMP_DIR)):
        fpath = os.path.join(COMP_DIR, f)
        size = os.path.getsize(fpath) if os.path.isfile(fpath) else 'dir'
        print(f'  {f} ({size})')

## 2. Dataset

Nucleotide vocabulary, sequence encoding, PDB/mmCIF parsing, and dataset classes.

In [ ]:
# Nucleotide vocabulary: A, C, G, U + padding + unknown
NUC_VOCAB = {'<pad>': 0, 'A': 1, 'C': 2, 'G': 3, 'U': 4, '<unk>': 5}
VOCAB_SIZE = len(NUC_VOCAB)

# Maximum reasonable coordinate value in angstroms for RNA structures
MAX_COORD_ANGSTROM = 1000.0


def encode_sequence(seq):
    """Encode an RNA sequence string to integer tokens."""
    return [NUC_VOCAB.get(c.upper(), NUC_VOCAB['<unk>']) for c in seq]


class RNATrainDataset(Dataset):
    """
    Training dataset using competition CSV files.

    Args:
        sequences_csv: CSV with columns [target_id, sequence, ...]
        labels_csv: CSV with per-residue 3D coordinates
                    Expected columns: [ID, resname, resid, x_1, y_1, z_1, ...]
                    ID format: {target_id}_{resid}
        max_seq_len: maximum sequence length (longer sequences are skipped)
    """

    def __init__(self, sequences_csv, labels_csv, max_seq_len=512):
        self.max_seq_len = max_seq_len

        # Load sequences
        seq_df = pd.read_csv(sequences_csv)
        assert 'target_id' in seq_df.columns and 'sequence' in seq_df.columns

        # Filter by length
        seq_df = seq_df[seq_df['sequence'].str.len() <= max_seq_len].reset_index(drop=True)
        self.sequences = dict(zip(seq_df['target_id'], seq_df['sequence']))

        # Load labels (low_memory=False to avoid mixed-type column issues)
        labels_df = pd.read_csv(labels_csv, low_memory=False)
        print(f'Labels CSV columns: {labels_df.columns.tolist()[:10]}... ({len(labels_df.columns)} total)')
        print(f'Labels CSV shape: {labels_df.shape}')

        # Filter to copy=1 only if copy column exists
        if 'copy' in labels_df.columns:
            n_before = len(labels_df)
            labels_df = labels_df[labels_df['copy'] == 1].reset_index(drop=True)
            print(f'Filtered to copy=1: {n_before} -> {len(labels_df)} rows')

        # Extract target_id from the ID column (format: targetid_resid)
        labels_df['target_id'] = labels_df['ID'].str.rsplit('_', n=1).str[0]

        # Detect coordinate columns — always use first prediction set (x_1, y_1, z_1)
        all_cols = labels_df.columns.tolist()
        if 'x_1' in all_cols and 'y_1' in all_cols and 'z_1' in all_cols:
            self.coord_cols = ['x_1', 'y_1', 'z_1']
        elif 'x' in all_cols and 'y' in all_cols and 'z' in all_cols:
            self.coord_cols = ['x', 'y', 'z']
        else:
            x_cols = [c for c in all_cols if c.startswith('x_')]
            y_cols = [c for c in all_cols if c.startswith('y_')]
            z_cols = [c for c in all_cols if c.startswith('z_')]
            if x_cols and y_cols and z_cols:
                self.coord_cols = [x_cols[0], y_cols[0], z_cols[0]]
            else:
                raise ValueError(f'Cannot find coordinate columns in {all_cols}')

        print(f'Coordinate columns used: {self.coord_cols}')

        # Force coordinate columns to numeric (handles mixed-type parsing)
        for col in self.coord_cols:
            labels_df[col] = pd.to_numeric(labels_df[col], errors='coerce')

        # Group labels by target_id, keep only targets we have sequences for
        self.labels = {}
        for tid, group in labels_df.groupby('target_id'):
            if tid in self.sequences:
                group = group.sort_values('resid')
                coords = group[self.coord_cols].values.astype(np.float32)
                self.labels[tid] = coords

        # Final target list: must have both sequence and labels
        self.target_ids = [tid for tid in self.sequences if tid in self.labels]
        print(f'Loaded {len(self.target_ids)} targets with sequences and labels')

        # Print coordinate statistics for debugging (exclude sentinel/extreme values)
        sample_tids = self.target_ids[:min(50, len(self.target_ids))]
        all_coords = np.concatenate([self.labels[tid] for tid in sample_tids])
        n_total = len(all_coords)
        n_nan = np.isnan(all_coords).any(axis=1).sum()
        valid_mask = (
            np.isfinite(all_coords).all(axis=1)
            & (np.abs(all_coords) <= MAX_COORD_ANGSTROM).all(axis=1)
        )
        valid = all_coords[valid_mask]
        n_extreme = n_total - n_nan - len(valid)
        if len(valid) > 0:
            print(f'Coord stats ({len(sample_tids)} targets, {len(valid)}/{n_total} valid residues): '
                  f'min={valid.min():.1f} max={valid.max():.1f} '
                  f'mean={valid.mean():.1f} std={valid.std():.1f}')
            print(f'  Excluded: {n_nan} NaN, {n_extreme} extreme/sentinel')

    def __len__(self):
        return len(self.target_ids)

    def __getitem__(self, idx):
        target_id = self.target_ids[idx]
        seq = self.sequences[target_id]

        tokens = encode_sequence(seq)
        seq_len = len(tokens)
        padded = tokens + [0] * (self.max_seq_len - seq_len)
        mask = [1] * seq_len + [0] * (self.max_seq_len - seq_len)

        # Load coordinates — always (L, 3) since we select exactly 3 columns
        raw_coords = self.labels[target_id]  # (n_residues, 3)
        coords = np.zeros((self.max_seq_len, 3), dtype=np.float32)
        n = min(len(raw_coords), seq_len, self.max_seq_len)
        coords[:n] = raw_coords[:n]

        # Handle invalid coordinates: NaN, Inf, or sentinel values (like -1e18)
        invalid = (
            np.isnan(coords).any(axis=1)
            | ~np.isfinite(coords).all(axis=1)
            | (np.abs(coords) > MAX_COORD_ANGSTROM).any(axis=1)
        )
        coords[invalid] = 0.0
        mask = np.array(mask, dtype=np.int64)
        mask[invalid] = 0

        # Center coordinates around centroid of valid residues
        # This removes the translation degree of freedom and gives the model
        # a better starting point (initial outputs near 0 ≈ centroid)
        valid_mask = mask.astype(bool)
        if valid_mask.any():
            centroid = coords[valid_mask].mean(axis=0)
            coords[valid_mask] -= centroid

        return {
            'tokens': torch.tensor(padded, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool),
            'coords': torch.tensor(coords, dtype=torch.float32),
            'seq_len': seq_len,
            'target_id': target_id,
        }


class RNATestDataset(Dataset):
    """Test dataset: RNA sequences only (no labels)."""

    def __init__(self, csv_path, max_seq_len=512):
        self.max_seq_len = max_seq_len
        self.df = pd.read_csv(csv_path)
        assert 'target_id' in self.df.columns
        assert 'sequence' in self.df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row['sequence']
        target_id = row['target_id']

        tokens = encode_sequence(seq)
        seq_len = len(tokens)
        if seq_len > self.max_seq_len:
            tokens = tokens[:self.max_seq_len]
            seq_len = self.max_seq_len

        padded = tokens + [0] * (self.max_seq_len - seq_len)
        mask = [1] * seq_len + [0] * (self.max_seq_len - seq_len)

        result = {
            'tokens': torch.tensor(padded, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool),
            'seq_len': seq_len,
            'target_id': target_id,
            'sequence': seq,
        }
        if 'description' in self.df.columns:
            result['description'] = row.get('description', '')
        if 'all_sequences' in self.df.columns:
            result['all_sequences'] = row.get('all_sequences', '')
        return result


print(f'Dataset classes defined. Vocab size: {VOCAB_SIZE}')

## 3. Model

Transformer-based architecture for predicting 3D C1' atom coordinates from RNA sequences.

Architecture:
1. Sequence Embedding: nucleotide tokens + positional encoding
2. Pairwise Representation: outer product of single representations
3. Transformer Encoder: self-attention with pairwise bias
4. Structure Module: predicts 3D coordinates via MLP heads
5. Multi-prediction Head: outputs 5 coordinate sets for ensemble scoring

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class PairwiseModule(nn.Module):
    """Computes pairwise representations via outer product, then refines with MLP."""

    def __init__(self, d_model, d_pair=64):
        super().__init__()
        self.proj_left = nn.Linear(d_model, d_pair)
        self.proj_right = nn.Linear(d_model, d_pair)
        self.pair_norm = nn.LayerNorm(d_pair)
        self.pair_mlp = nn.Sequential(
            nn.Linear(d_pair, d_pair * 2), nn.GELU(), nn.Linear(d_pair * 2, d_pair)
        )
        self.pair_to_bias = nn.Linear(d_pair, 1)

    def forward(self, single_repr, mask):
        left = self.proj_left(single_repr)
        right = self.proj_right(single_repr)
        pair_repr = torch.einsum('bid,bjd->bijd', left, right)
        pair_repr = self.pair_norm(pair_repr)
        pair_repr = pair_repr + self.pair_mlp(pair_repr)
        pair_mask = mask.unsqueeze(-1) * mask.unsqueeze(-2)
        pair_repr = pair_repr * pair_mask.unsqueeze(-1)
        attn_bias = self.pair_to_bias(pair_repr).squeeze(-1)
        return pair_repr, attn_bias


class StructureAwareAttention(nn.Module):
    """Multi-head self-attention with pairwise bias."""

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = self.d_head ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask, attn_bias=None):
        B, L, D = x.shape
        q = rearrange(self.q_proj(x), 'b l (h d) -> b h l d', h=self.n_heads)
        k = rearrange(self.k_proj(x), 'b l (h d) -> b h l d', h=self.n_heads)
        v = rearrange(self.v_proj(x), 'b l (h d) -> b h l d', h=self.n_heads)
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if attn_bias is not None:
            attn = attn + attn_bias.unsqueeze(1)
        mask_2d = mask.unsqueeze(1).unsqueeze(2)
        attn = attn.masked_fill(~mask_2d, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h l d -> b l (h d)')
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = StructureAwareAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, mask, attn_bias=None):
        x = x + self.attn(self.norm1(x), mask, attn_bias)
        x = x + self.ffn(self.norm2(x))
        return x


class StructureModule(nn.Module):
    """Predicts 3D coordinates and per-residue confidence."""

    def __init__(self, d_model, num_predictions=5):
        super().__init__()
        self.num_predictions = num_predictions
        self.coord_heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model), nn.GELU(),
                nn.Linear(d_model, d_model // 2), nn.GELU(),
                nn.Linear(d_model // 2, 3),
            ) for _ in range(num_predictions)
        ])
        self.confidence_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, num_predictions), nn.Sigmoid(),
        )

    def forward(self, single_repr):
        coords = torch.stack([head(single_repr) for head in self.coord_heads], dim=2)
        confidence = self.confidence_head(single_repr)
        return {'coords': coords, 'confidence': confidence}


class RNAFoldModel(nn.Module):
    """End-to-end RNA 3D structure prediction model with recycling."""

    def __init__(self, d_model=256, n_heads=8, n_layers=8, d_ff=1024,
                 dropout=0.1, num_predictions=5, max_seq_len=512, n_recycles=1):
        super().__init__()
        self.d_model = d_model
        self.num_predictions = num_predictions
        self.n_recycles = n_recycles
        self.token_emb = nn.Embedding(VOCAB_SIZE, d_model, padding_idx=0)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len=max_seq_len + 100)
        self.pairwise = PairwiseModule(d_model, d_pair=64)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.structure_module = StructureModule(d_model, num_predictions)
        # Recycling: project previous coordinates back into embedding space
        if n_recycles > 0:
            self.coord_proj = nn.Linear(3, d_model)
        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if p.dim() > 1 and 'token_emb' not in name:
                nn.init.xavier_uniform_(p)
        with torch.no_grad():
            self.token_emb.weight[0].zero_()

    def forward(self, tokens, mask):
        x_base = self.token_emb(tokens)
        x_base = self.pos_enc(x_base)

        prev_coords = None
        for r in range(1 + self.n_recycles):
            # Add previous coordinate information on recycle iterations
            if r > 0 and prev_coords is not None:
                x = x_base + self.coord_proj(prev_coords.detach())
            else:
                x = x_base

            # Recompute pairwise features from updated single representation
            _, attn_bias = self.pairwise(x, mask)

            for layer in self.layers:
                x = x * mask.unsqueeze(-1).float()
                x = layer(x, mask, attn_bias)

            output = self.structure_module(x)
            # Use first prediction head for recycling input
            prev_coords = output['coords'][:, :, 0, :]

        return output

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print('Model defined successfully.')

## 4. Loss Functions and Metrics

FAPE loss, distance matrix loss, TM-score, and combined training loss.

In [ ]:
def compute_distance_matrix(coords):
    """Compute pairwise distance matrix. coords: (B, L, 3) -> (B, L, L)"""
    diff = coords.unsqueeze(2) - coords.unsqueeze(1)
    return torch.sqrt((diff ** 2).sum(-1) + 1e-8)


def distance_matrix_loss(pred_coords, true_coords, mask):
    """L1 loss on pairwise distance matrices (rotationally invariant)."""
    pred_dist = compute_distance_matrix(pred_coords)
    true_dist = compute_distance_matrix(true_coords)
    pair_mask = (mask.unsqueeze(-1) * mask.unsqueeze(-2)).float()
    loss = (pred_dist - true_dist).abs() * pair_mask
    return loss.sum() / (pair_mask.sum() + 1e-8)


def local_distance_loss(pred_coords, true_coords, mask, max_sep=4):
    """L1 loss on local pairwise distances (consecutive residues up to max_sep apart).
    Captures backbone geometry and local structure."""
    B, L, _ = pred_coords.shape
    total_loss = 0.0
    total_count = 0.0
    for sep in range(1, max_sep + 1):
        pred_d = torch.sqrt(((pred_coords[:, sep:] - pred_coords[:, :-sep]) ** 2).sum(-1) + 1e-8)
        true_d = torch.sqrt(((true_coords[:, sep:] - true_coords[:, :-sep]) ** 2).sum(-1) + 1e-8)
        pair_mask = (mask[:, sep:] * mask[:, :-sep]).float()
        total_loss += ((pred_d - true_d).abs() * pair_mask).sum()
        total_count += pair_mask.sum()
    return total_loss / (total_count + 1e-8)


def kabsch_aligned_loss(pred_coords, true_coords, mask, clamp=50.0):
    """Coordinate L2 loss after Kabsch alignment (rotation-invariant).

    Rotation matrix is detached for gradient stability (no backprop through SVD).
    Gradients flow through the predicted coordinates after alignment.
    """
    B = pred_coords.shape[0]
    total_loss = 0.0
    n_valid = 0

    for b in range(B):
        m = mask[b]
        if m.sum() < 3:
            continue
        pred_b = pred_coords[b][m]  # (N, 3)
        true_b = true_coords[b][m]  # (N, 3)

        # Center
        pred_mean = pred_b.mean(0, keepdim=True)
        true_mean = true_b.mean(0, keepdim=True)
        pred_c = pred_b - pred_mean
        true_c = true_b - true_mean

        # Kabsch rotation (detached — no gradient through SVD)
        with torch.no_grad():
            H = pred_c.T @ true_c  # (3, 3)
            try:
                U, S, Vh = torch.linalg.svd(H)
                d = torch.det(Vh.T @ U.T)
                diag = torch.ones(3, device=pred_b.device)
                diag[2] = d.sign()
                R = (Vh.T * diag.unsqueeze(0)) @ U.T  # (3, 3)
            except Exception:
                R = torch.eye(3, device=pred_b.device)

        # Apply rotation (gradient flows through pred_c)
        aligned = pred_c @ R.T  # (N, 3)

        # Per-residue L2 distance
        dists = torch.sqrt(((aligned - true_c) ** 2).sum(-1) + 1e-8)
        if clamp > 0:
            dists = torch.clamp(dists, max=clamp)
        total_loss += dists.mean()
        n_valid += 1

    return total_loss / max(n_valid, 1)


def kabsch_align(pred, true):
    """Align pred to true using Kabsch algorithm (numpy, for TM-score eval)."""
    pred_center = pred.mean(axis=0)
    true_center = true.mean(axis=0)
    pred_c = pred - pred_center
    true_c = true - true_center
    H = pred_c.T @ true_c
    if not np.isfinite(H).all():
        return pred
    try:
        U, S, Vt = np.linalg.svd(H)
    except np.linalg.LinAlgError:
        return pred
    d = np.linalg.det(Vt.T @ U.T)
    sign_matrix = np.diag([1, 1, np.sign(d)])
    R = Vt.T @ sign_matrix @ U.T
    return pred_c @ R.T + true_center


def compute_tm_score(pred_coords, true_coords):
    """Compute TM-score between predicted and true structures."""
    L = len(true_coords)
    if L == 0:
        return 0.0
    if not np.isfinite(pred_coords).all():
        return 0.0
    d0 = max(0.6 * (L - 0.5) ** (1.0 / 3.0) - 2.5, 0.5)
    aligned = kabsch_align(pred_coords, true_coords)
    dists = np.sqrt(((aligned - true_coords) ** 2).sum(axis=1))
    return float((1.0 / (1.0 + (dists / d0) ** 2)).sum() / L)


def best_of_n_tm_score(pred_coords_list, true_coords):
    """Compute best-of-N TM-score (competition metric)."""
    return max(compute_tm_score(pred, true_coords) for pred in pred_coords_list)


class CombinedLoss(nn.Module):
    """Combined training loss with curriculum: distance losses first, add aligned later.

    Phase 1 (epochs 0 to aligned_warmup): only distance matrix + local distance
    Phase 2 (after aligned_warmup): add Kabsch-aligned coordinate loss
    """

    def __init__(self, dist_weight=1.0, local_weight=2.0, aligned_weight=0.5,
                 confidence_weight=0.1, clamp=50.0, aligned_warmup_epoch=10):
        super().__init__()
        self.dist_weight = dist_weight
        self.local_weight = local_weight
        self.aligned_weight = aligned_weight
        self.confidence_weight = confidence_weight
        self.clamp = clamp
        self.aligned_warmup_epoch = aligned_warmup_epoch
        self.current_epoch = 0

    def forward(self, pred, true_coords, mask):
        num_preds = pred['coords'].shape[2]
        total_dist = 0.0
        total_local = 0.0
        total_aligned = 0.0
        per_pred_losses = []

        use_aligned = self.current_epoch >= self.aligned_warmup_epoch

        for i in range(num_preds):
            pred_i = pred['coords'][:, :, i, :]
            d = distance_matrix_loss(pred_i, true_coords, mask)
            loc = local_distance_loss(pred_i, true_coords, mask)
            total_dist += d
            total_local += loc
            pred_loss = d + loc

            if use_aligned:
                a = kabsch_aligned_loss(pred_i, true_coords, mask, clamp=self.clamp)
                total_aligned += a
                pred_loss = pred_loss + a

            per_pred_losses.append(pred_loss.detach())

        total_dist /= num_preds
        total_local /= num_preds
        total_aligned /= num_preds

        confidence_loss = torch.tensor(0.0, device=true_coords.device)
        if self.confidence_weight > 0:
            per_pred_losses_t = torch.stack(per_pred_losses)
            with torch.no_grad():
                quality = 1.0 / (1.0 + per_pred_losses_t)
                quality = quality / (quality.max() + 1e-8)
                quality_target = quality.unsqueeze(0).unsqueeze(0).expand_as(pred['confidence'])
            confidence_loss = F.mse_loss(
                pred['confidence'] * mask.unsqueeze(-1).float(),
                quality_target * mask.unsqueeze(-1).float(),
            )

        total_loss = (
            self.dist_weight * total_dist
            + self.local_weight * total_local
            + self.confidence_weight * confidence_loss
        )
        if use_aligned:
            total_loss = total_loss + self.aligned_weight * total_aligned

        return {
            'loss': total_loss,
            'dist_loss': total_dist,
            'local_loss': total_local,
            'aligned_loss': total_aligned,
            'confidence_loss': confidence_loss,
        }


print('Loss functions and metrics defined.')

## 5. Training Pipeline

Mixed precision training with warmup + cosine decay, gradient clipping, TM-score validation, and checkpointing.

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_lr_scheduler(optimizer, warmup_steps, total_steps):
    """Linear warmup then cosine decay."""
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, dataloader, optimizer, scheduler, criterion,
                    scaler, device, grad_clip, epoch):
    model.train()
    total_loss = 0
    total_aligned = 0
    total_dist = 0
    n_batches = 0
    nan_batches = 0
    use_amp = device.type == 'cuda'

    pbar = tqdm(dataloader, desc=f'Epoch {epoch}')
    for batch in pbar:
        tokens = batch['tokens'].to(device)
        mask = batch['mask'].to(device)
        coords = batch['coords'].to(device)

        if mask.sum() == 0:
            continue

        optimizer.zero_grad()

        with torch.autocast('cuda', enabled=use_amp):
            pred = model(tokens, mask)
            loss_dict = criterion(pred, coords, mask)
            loss = loss_dict['loss']

        if not torch.isfinite(loss):
            nan_batches += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        total_aligned += loss_dict['aligned_loss'].item()
        total_dist += loss_dict['dist_loss'].item()
        n_batches += 1

        pbar.set_postfix({
            'loss': f'{total_loss / n_batches:.4f}',
            'aligned': f'{total_aligned / n_batches:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}',
        })

    if nan_batches > 0:
        print(f'  Warning: skipped {nan_batches} batches with NaN/Inf loss')

    return {
        'loss': total_loss / max(n_batches, 1),
        'aligned_loss': total_aligned / max(n_batches, 1),
        'dist_loss': total_dist / max(n_batches, 1),
    }


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_tm = 0
    n_batches = 0
    n_tm = 0

    for batch in tqdm(dataloader, desc='Validating'):
        tokens = batch['tokens'].to(device)
        mask = batch['mask'].to(device)
        coords = batch['coords'].to(device)

        if mask.sum() == 0:
            continue

        pred = model(tokens, mask)
        loss_dict = criterion(pred, coords, mask)

        loss_val = loss_dict['loss'].item()
        if not np.isfinite(loss_val):
            continue

        total_loss += loss_val
        n_batches += 1

        pred_coords = pred['coords'].cpu().numpy()
        true_coords = coords.cpu().numpy()
        masks = mask.cpu().numpy()

        for b in range(pred_coords.shape[0]):
            seq_len = masks[b].sum().astype(int)
            if seq_len < 3:
                continue
            true_c = true_coords[b, :seq_len]
            if np.abs(true_c).sum() < 1e-6:
                continue
            pred_list = [pred_coords[b, :seq_len, i] for i in range(pred_coords.shape[2])]
            tm = max(compute_tm_score(p, true_c) for p in pred_list)
            total_tm += tm
            n_tm += 1

    return {
        'loss': total_loss / max(n_batches, 1),
        'tm_score': total_tm / max(n_tm, 1),
    }


print('Training pipeline defined.')

## 6. Train the Model

Run training with the configuration above. Skip this section if loading pretrained weights.

In [ ]:
# Set to True to run training, False to skip to inference
RUN_TRAINING = False

if RUN_TRAINING:
    cfg = CONFIG
    set_seed(cfg['training']['seed'])
    os.makedirs(cfg['training']['save_dir'], exist_ok=True)

    # Check that training data exists
    train_seq = cfg['data']['train_sequences_csv']
    train_lbl = cfg['data']['train_labels_csv']

    if not os.path.exists(train_seq):
        print(f'WARNING: {train_seq} not found. Skipping training.')
    elif not os.path.exists(train_lbl):
        print(f'WARNING: {train_lbl} not found. Skipping training.')
    else:
        # Training dataset
        train_dataset = RNATrainDataset(
            sequences_csv=train_seq,
            labels_csv=train_lbl,
            max_seq_len=cfg['data']['max_seq_len'],
        )

        if len(train_dataset) == 0:
            print('WARNING: Training dataset is empty. Skipping training.')
        else:
            train_loader = DataLoader(
                train_dataset, batch_size=cfg['training']['batch_size'],
                shuffle=True, num_workers=cfg['data']['num_workers'], pin_memory=True,
            )

            # Validation dataset (use competition validation split if available)
            val_seq = cfg['data']['val_sequences_csv']
            val_lbl = cfg['data']['val_labels_csv']
            if os.path.exists(val_seq) and os.path.exists(val_lbl):
                val_dataset = RNATrainDataset(
                    sequences_csv=val_seq,
                    labels_csv=val_lbl,
                    max_seq_len=cfg['data']['max_seq_len'],
                )
                print(f'Using competition validation split: {len(val_dataset)} targets')
            else:
                # Fall back to random split from training data
                n_val = max(1, len(train_dataset) // 10)
                n_train = len(train_dataset) - n_val
                train_dataset, val_dataset = torch.utils.data.random_split(
                    train_dataset, [n_train, n_val]
                )
                train_loader = DataLoader(
                    train_dataset, batch_size=cfg['training']['batch_size'],
                    shuffle=True, num_workers=cfg['data']['num_workers'], pin_memory=True,
                )
                print(f'Using random 90/10 split: {n_train} train, {n_val} val')

            val_loader = DataLoader(
                val_dataset, batch_size=cfg['training']['batch_size'],
                shuffle=False, num_workers=0,
            )

            # Model
            model = RNAFoldModel(
                d_model=cfg['model']['d_model'],
                n_heads=cfg['model']['n_heads'],
                n_layers=cfg['model']['n_layers'],
                d_ff=cfg['model']['d_ff'],
                dropout=cfg['model']['dropout'],
                num_predictions=cfg['model']['num_predictions'],
                max_seq_len=cfg['data']['max_seq_len'],
                n_recycles=cfg['model']['n_recycles'],
            ).to(device)
            print(f'Model parameters: {model.count_parameters():,}')

            # Optimizer + scheduler
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=cfg['training']['learning_rate'],
                weight_decay=cfg['training']['weight_decay'],
            )
            total_steps = cfg['training']['num_epochs'] * len(train_loader)
            scheduler = get_lr_scheduler(optimizer, cfg['training']['warmup_steps'], total_steps)
            criterion = CombinedLoss()
            scaler = GradScaler('cuda', enabled=device.type == 'cuda')

            best_tm = 0.0

            for epoch in range(cfg['training']['num_epochs']):
                t0 = time.time()
                criterion.current_epoch = epoch
                train_metrics = train_one_epoch(
                    model, train_loader, optimizer, scheduler, criterion,
                    scaler, device, cfg['training']['grad_clip'], epoch,
                )
                val_metrics = validate(model, val_loader, criterion, device)
                elapsed = time.time() - t0

                print(
                    f'Epoch {epoch}: '
                    f'train_loss={train_metrics["loss"]:.4f} '
                    f'val_loss={val_metrics["loss"]:.4f} '
                    f'val_tm={val_metrics["tm_score"]:.4f} '
                    f'time={elapsed:.1f}s'
                )

                ckpt = {
                    'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'epoch': epoch,
                    'best_tm': best_tm,
                    'config': cfg,
                }

                if val_metrics['tm_score'] > best_tm:
                    best_tm = val_metrics['tm_score']
                    ckpt['best_tm'] = best_tm
                    torch.save(ckpt, os.path.join(cfg['training']['save_dir'], 'best_model.pt'))
                    print(f'  New best TM-score: {best_tm:.4f}')

                torch.save(ckpt, os.path.join(cfg['training']['save_dir'], 'last_model.pt'))

            print(f'Training complete. Best TM-score: {best_tm:.4f}')
else:
    print('Training skipped. Set RUN_TRAINING = True to train.')

## 7. Load Model and Test Data

In [ ]:
# Load test sequences (or create sample data for local testing)
test_csv_path = CONFIG['data']['test_csv']

if os.path.exists(test_csv_path):
    test_df = pd.read_csv(test_csv_path)
    print(f'Loaded test_sequences.csv: {len(test_df)} targets')
else:
    print(f'test_sequences.csv not found at {test_csv_path}')
    print('Creating sample test data for local development...')
    sample_data = {
        'target_id': ['SAMPLE_001', 'SAMPLE_002', 'SAMPLE_003'],
        'sequence': [
            'GGGCGAUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA',
            'GGUCCGAGCAGAAGACGGCUACCCAUUCCGAUUGAGUCCUAGAAAGCUUCUUCUUUAAUUUU',
            'GCGACCGGGGCUGGCUUGGUAAUGGUACUCCCCUGUCACGGGAGAGAAUGUGGGUUCAAAUCCCAUCGGUCGCGCCA',
        ],
    }
    test_df = pd.DataFrame(sample_data)
    os.makedirs(os.path.dirname(test_csv_path), exist_ok=True)
    test_df.to_csv(test_csv_path, index=False)
    print(f'Sample data saved to {test_csv_path}')

print(f'Test targets: {len(test_df)}')
print(f'Sequence lengths: {test_df["sequence"].str.len().tolist()}')
print(test_df.head())

In [ ]:
# Load model for inference (dropout=0)
MODEL_CONFIG = CONFIG['model'].copy()
MODEL_CONFIG['dropout'] = 0.0
MODEL_CONFIG['max_seq_len'] = CONFIG['data']['max_seq_len']

model = RNAFoldModel(**MODEL_CONFIG).to(device)

ckpt_path = os.path.join(MODEL_DIR, 'best_model.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print('Loaded trained model weights.')
else:
    print('WARNING: No checkpoint found. Using random weights.')

model.eval()
print(f'Model params: {model.count_parameters():,}')

## 8. Template-Based Modeling (TBM) Pipeline

Primary prediction method: find similar sequences in the training set via pairwise alignment,
adapt their known 3D coordinates to the query sequence.
Falls back to our trained transformer model for sequences with no good templates.

In [ ]:
# ─── RNA geometry constants ───
RNA_BOND_LENGTH = 5.95   # typical C1'-C1' distance between consecutive nucleotides
RNA_I2_LENGTH = 10.2     # typical C1'-C1' distance for i to i+2


def load_training_coords(labels_csv_path):
    """Load training label coordinates, returning {target_id: np.ndarray(N,3)}."""
    labels_df = pd.read_csv(labels_csv_path, low_memory=False)

    if 'copy' in labels_df.columns:
        labels_df = labels_df[labels_df['copy'] == 1].reset_index(drop=True)

    # Detect coordinate columns
    cols = labels_df.columns.tolist()
    if 'x_1' in cols and 'y_1' in cols and 'z_1' in cols:
        coord_cols = ['x_1', 'y_1', 'z_1']
    elif 'x' in cols and 'y' in cols and 'z' in cols:
        coord_cols = ['x', 'y', 'z']
    else:
        x_cols = sorted(c for c in cols if c.startswith('x_'))
        y_cols = sorted(c for c in cols if c.startswith('y_'))
        z_cols = sorted(c for c in cols if c.startswith('z_'))
        coord_cols = [x_cols[0], y_cols[0], z_cols[0]]

    for col in coord_cols:
        labels_df[col] = pd.to_numeric(labels_df[col], errors='coerce')
    labels_df['resid'] = pd.to_numeric(labels_df['resid'], errors='coerce')
    labels_df['target_id'] = labels_df['ID'].astype(str).str.rsplit('_', n=1).str[0]

    coord_map = {}
    for target_id, group in labels_df.groupby('target_id'):
        ordered = group.sort_values('resid')
        coords = ordered[coord_cols].values.astype(np.float32)
        valid = np.isfinite(coords).all(axis=1)
        coords = coords[valid]
        if len(coords) > 0:
            coord_map[target_id] = coords
    return coord_map


# ─── k-mer based shortlisting for fast template search ───

def build_kmers(sequence, k=3):
    """Build set of k-mers from a sequence for fast similarity pre-filtering."""
    if len(sequence) < k:
        return frozenset([sequence]) if sequence else frozenset()
    return frozenset(sequence[i:i+k] for i in range(len(sequence) - k + 1))


def jaccard_score(left, right):
    if not left or not right:
        return 0.0
    inter = len(left & right)
    union = len(left | right)
    return inter / union if union else 0.0


class TemplateIndex:
    """Fast template lookup using k-mer Jaccard pre-filtering."""

    def __init__(self, sequence_df, coord_map, k=3):
        self.entries = []
        for _, row in sequence_df.iterrows():
            tid = row['target_id']
            if tid not in coord_map:
                continue
            seq = str(row['sequence'])
            self.entries.append({
                'target_id': tid,
                'sequence': seq,
                'coords': coord_map[tid],
                'kmers': build_kmers(seq, k=k),
            })

    def shortlist(self, query_seq, shortlist_size):
        """Return top-N most similar templates by k-mer Jaccard + length filter."""
        query_kmers = build_kmers(query_seq)
        scored = []
        for entry in self.entries:
            len_delta = abs(len(entry['sequence']) - len(query_seq)) / max(len(entry['sequence']), len(query_seq), 1)
            if len_delta > 0.40:
                continue
            scored.append((jaccard_score(query_kmers, entry['kmers']), -len_delta, entry))
        scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
        return [e for _, _, e in scored[:shortlist_size]]


# ─── Pairwise aligner setup (BioPython) ───

def _set_aligner_attr(aligner, names, value):
    """Set attribute on aligner, trying multiple names for compatibility."""
    for name in names:
        if hasattr(aligner, name):
            setattr(aligner, name, value)
            return
    raise AttributeError(f"Could not set any of {names!r} on PairwiseAligner.")


def make_aligner():
    aligner = PairwiseAligner()
    aligner.mode = 'global'
    aligner.match_score = 2.0
    aligner.mismatch_score = -1.5
    _set_aligner_attr(aligner, ['open_gap_score'], -8.0)
    _set_aligner_attr(aligner, ['extend_gap_score'], -0.4)
    _set_aligner_attr(aligner, ['open_left_deletion_score', 'query_left_open_gap_score'], -8.0)
    _set_aligner_attr(aligner, ['extend_left_deletion_score', 'query_left_extend_gap_score'], -0.4)
    _set_aligner_attr(aligner, ['open_right_deletion_score', 'query_right_open_gap_score'], -8.0)
    _set_aligner_attr(aligner, ['extend_right_deletion_score', 'query_right_extend_gap_score'], -0.4)
    _set_aligner_attr(aligner, ['open_left_insertion_score', 'target_left_open_gap_score'], -8.0)
    _set_aligner_attr(aligner, ['extend_left_insertion_score', 'target_left_extend_gap_score'], -0.4)
    _set_aligner_attr(aligner, ['open_right_insertion_score', 'target_right_open_gap_score'], -8.0)
    _set_aligner_attr(aligner, ['extend_right_insertion_score', 'target_right_extend_gap_score'], -0.4)
    return aligner


if HAS_BIOPYTHON:
    _GLOBAL_ALIGNER = make_aligner()
else:
    _GLOBAL_ALIGNER = None


# ─── Template hit finding with full alignment scoring ───

def find_template_hits(query_seq, template_index, aligner, shortlist_size=128, top_n=30):
    """Find top template hits using k-mer shortlisting then full pairwise alignment."""
    hits = []
    for entry in template_index.shortlist(query_seq, shortlist_size):
        tseq = entry['sequence']
        len_delta = abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq), 1)
        if len_delta > 0.30:
            continue
        aln = next(iter(aligner.align(query_seq, tseq)))
        similarity = aln.score / (2.0 * min(len(query_seq), len(tseq)))
        identical = sum(
            1
            for (qs, qe), (ts, te) in zip(*aln.aligned)
            for qp, tp in zip(range(qs, qe), range(ts, te))
            if query_seq[qp] == tseq[tp]
        )
        pct_id = 100.0 * identical / max(len(query_seq), 1)
        hits.append({
            'target_id': entry['target_id'],
            'sequence': tseq,
            'similarity': similarity,
            'percent_identity': pct_id,
            'coords': entry['coords'],
        })
    hits.sort(key=lambda x: (x['similarity'], x['percent_identity']), reverse=True)
    return hits[:top_n]


# ─── Coordinate adaptation: map template coords to query via alignment ───

def adapt_template_to_query(query_seq, template_seq, template_coords):
    """Map template 3D coordinates onto query sequence positions via alignment."""
    aln = next(iter(_GLOBAL_ALIGNER.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan, dtype=np.float64)

    for (qs, qe), (ts, te) in zip(*aln.aligned):
        chunk = template_coords[ts:te]
        if len(chunk) == (qe - qs):
            new_coords[qs:qe] = chunk

    # Interpolate gaps
    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            prev = next((j for j in range(i-1, -1, -1) if not np.isnan(new_coords[j, 0])), -1)
            nxt = next((j for j in range(i+1, len(new_coords)) if not np.isnan(new_coords[j, 0])), -1)
            if prev >= 0 and nxt >= 0:
                w = (i - prev) / max(nxt - prev, 1)
                new_coords[i] = (1.0 - w) * new_coords[prev] + w * new_coords[nxt]
            elif prev >= 0:
                new_coords[i] = new_coords[prev] + [3.0, 0.0, 0.0]
            elif nxt >= 0:
                new_coords[i] = new_coords[nxt] + [3.0, 0.0, 0.0]
            else:
                new_coords[i] = [i * 3.0, 0.0, 0.0]

    return np.nan_to_num(new_coords, nan=0.0).astype(np.float32)


# ─── Chain segment parsing ───

def parse_stoichiometry(stoich):
    if pd.isna(stoich) or str(stoich).strip() == '':
        return []
    return [(ch.strip(), int(cnt)) for part in str(stoich).split(';')
            for ch, cnt in [part.split(':')]]


def parse_fasta(fasta_content):
    out, cur, parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if cur is not None:
                out[cur] = ''.join(parts)
            cur = line[1:].split()[0]
            parts = []
        else:
            parts.append(line.replace(' ', ''))
    if cur is not None:
        out[cur] = ''.join(parts)
    return out


def get_chain_segments(row):
    """Parse chain segments from stoichiometry/all_sequences columns."""
    seq = str(row['sequence'])
    stoich = row.get('stoichiometry', '')
    all_seq = row.get('all_sequences', '')
    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip() == '' or str(all_seq).strip() == '':
        return [(0, len(seq))]
    try:
        chain_map = parse_fasta(all_seq)
        order = parse_stoichiometry(stoich)
        segs, pos = [], 0
        for ch, cnt in order:
            base = chain_map.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                segs.append((pos, pos + len(base)))
                pos += len(base)
        return segs if pos == len(seq) else [(0, len(seq))]
    except Exception:
        return [(0, len(seq))]


def build_segments_map(df):
    return {row['target_id']: get_chain_segments(row) for _, row in df.iterrows()}


# ─── RNA geometry constraint refinement ───

def adaptive_rna_constraints(coords, segments=None, confidence=1.0, passes=2):
    """Refine coordinates with RNA-specific geometry constraints.

    Enforces:
    - C1'-C1' bond distances (~5.95 A)
    - i to i+2 distances (~10.2 A)
    - Laplacian smoothing
    - Self-avoidance (min 3.2 A between non-adjacent residues)
    """
    out = coords.astype(np.float64, copy=True)
    if segments is None:
        segments = [(0, len(out))]
    strength = max(0.75 * (1.0 - min(confidence, 0.97)), 0.02)

    for _ in range(passes):
        for s, e in segments:
            chunk = out[s:e]
            L = e - s
            if L < 3:
                continue

            # Bond length correction (i to i+1, target ~5.95 A)
            d = chunk[1:] - chunk[:-1]
            dn = np.linalg.norm(d, axis=1) + 1e-6
            adj = d * ((RNA_BOND_LENGTH - dn) / dn)[:, None] * (0.22 * strength)
            chunk[:-1] -= adj
            chunk[1:] += adj

            # i to i+2 distance correction (~10.2 A)
            d2 = chunk[2:] - chunk[:-2]
            d2n = np.linalg.norm(d2, axis=1) + 1e-6
            adj2 = d2 * ((RNA_I2_LENGTH - d2n) / d2n)[:, None] * (0.10 * strength)
            chunk[:-2] -= adj2
            chunk[2:] += adj2

            # Laplacian smoothing
            chunk[1:-1] += (0.06 * strength) * (0.5 * (chunk[:-2] + chunk[2:]) - chunk[1:-1])

            # Self-avoidance
            if L >= 25:
                idx = np.linspace(0, L-1, min(L, 160)).astype(int) if L > 220 else np.arange(L)
                P = chunk[idx]
                diff = P[:, None, :] - P[None, :, :]
                dm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (dm < 3.2)
                if np.any(mask):
                    vec = (diff * ((3.2 - dm) / dm)[:, :, None] * mask[:, :, None]).sum(axis=1)
                    chunk[idx] += (0.015 * strength) * vec

            out[s:e] = chunk
    return out.astype(np.float32)


# ─── Diversity transforms for generating multiple predictions from one template ───

def _rotmat(axis, angle):
    """Rotation matrix from axis-angle."""
    a = np.asarray(axis, dtype=np.float64)
    a = a / (np.linalg.norm(a) + 1e-12)
    x, y, z = a
    c, s = np.cos(angle), np.sin(angle)
    cc = 1.0 - c
    return np.array([
        [c + x*x*cc, x*y*cc - z*s, x*z*cc + y*s],
        [y*x*cc + z*s, c + y*y*cc, y*z*cc - x*s],
        [z*x*cc - y*s, z*y*cc + x*s, c + z*z*cc],
    ], dtype=np.float64)


def apply_hinge(coords, seg, rng, deg=22.0):
    """Apply random hinge rotation at a pivot point within a chain segment."""
    s, e = seg
    if e - s < 30:
        return coords.astype(np.float32)
    pivot = s + int(rng.integers(10, (e - s) - 10))
    R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
    out = coords.astype(np.float64, copy=True)
    p0 = out[pivot].copy()
    out[pivot+1:e] = (out[pivot+1:e] - p0) @ R.T + p0
    return out.astype(np.float32)


def jitter_chains(coords, segs, rng, deg=12.0, trans=1.5):
    """Apply random rotation + translation to each chain segment."""
    out = coords.astype(np.float64, copy=True)
    gc = out.mean(axis=0, keepdims=True)
    for s, e in segs:
        R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
        shift = rng.normal(size=3)
        shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0, trans))
        c = out[s:e].mean(axis=0, keepdims=True)
        out[s:e] = (out[s:e] - c) @ R.T + c + shift
    out -= out.mean(axis=0, keepdims=True) - gc
    return out.astype(np.float32)


def smooth_wiggle(coords, segs, rng, amp=0.8):
    """Apply smooth random displacement using spline interpolation."""
    out = coords.astype(np.float64, copy=True)
    for s, e in segs:
        L = e - s
        if L < 20:
            continue
        ctrl = np.linspace(0, L-1, 6)
        disp = rng.normal(0, amp, size=(6, 3))
        t = np.arange(L)
        out[s:e] += np.vstack([np.interp(t, ctrl, disp[:, k]) for k in range(3)]).T
    return out.astype(np.float32)


def generate_rna_helix(sequence, seed=None):
    """Generate idealized A-form RNA helix — last-resort de-novo fallback."""
    rng = np.random.default_rng(seed)
    n = len(sequence)
    coords = np.zeros((n, 3), dtype=np.float32)
    for i in range(n):
        ang = i * 0.6
        radius = 10.0 + 0.35 * np.sin(i * 0.15)
        coords[i] = [
            radius * np.cos(ang),
            radius * np.sin(ang),
            i * 2.5 + float(rng.normal(0, 0.15)),
        ]
    return coords


# ─── Candidate quality scoring and deduplication ───

def is_collapsed(coords):
    """Check if coordinates are collapsed (all atoms in same place)."""
    if coords.ndim != 2 or coords.shape[0] < 2:
        return True
    if not np.isfinite(coords).all():
        return True
    diffs = np.linalg.norm(coords[1:] - coords[:-1], axis=1)
    if np.all(diffs < 1e-4):
        return True
    return float(np.mean(diffs < 0.25)) > 0.90


def candidate_geometry_score(coords, segments):
    """Score a candidate structure by RNA geometry quality."""
    if not np.isfinite(coords).all() or len(coords) < 3:
        return -1e9
    bond_penalty = 0.0
    overlap_penalty = 0.0
    for s, e in segments:
        chunk = coords[s:e]
        if len(chunk) < 3:
            continue
        bond = np.linalg.norm(chunk[1:] - chunk[:-1], axis=1)
        bond_penalty += np.mean(np.abs(bond - RNA_BOND_LENGTH))
        idx = np.arange(len(chunk))
        diff = chunk[:, None, :] - chunk[None, :, :]
        dist = np.linalg.norm(diff, axis=2) + 1e-6
        sep = np.abs(idx[:, None] - idx[None, :])
        overlap_mask = (sep > 2) & (dist < 2.6)
        overlap_penalty += float(overlap_mask.sum()) / max(len(chunk), 1)
    span = np.linalg.norm(coords.max(axis=0) - coords.min(axis=0))
    return -bond_penalty - 0.25 * overlap_penalty - 0.002 * max(span - 500.0, 0.0)


def dedupe_candidates(candidates, min_mean_distance=0.75):
    """Remove near-duplicate candidate structures."""
    kept = []
    for cand in candidates:
        centered = cand - cand.mean(axis=0, keepdims=True)
        is_dup = False
        for existing in kept:
            other = existing - existing.mean(axis=0, keepdims=True)
            mean_dist = np.linalg.norm(centered - other, axis=1).mean()
            if mean_dist < min_mean_distance:
                is_dup = True
                break
        if not is_dup:
            kept.append(cand)
    return kept


def rank_candidates(candidates, segments, desired_n):
    """Filter collapsed, deduplicate, and rank by geometry score."""
    if not candidates:
        return []
    filtered = [c for c in candidates if not is_collapsed(c)]
    if not filtered:
        return []
    filtered = dedupe_candidates(filtered)
    filtered.sort(key=lambda arr: candidate_geometry_score(arr, segments), reverse=True)
    return filtered[:desired_n]


# ─── TBM prediction pipeline ───

def tbm_predict(query_seq, template_index, segments, row_idx=0, seed=42):
    """Generate up to N_PREDICTIONS structure predictions via template-based modeling.

    For each template hit:
    - Adapt template coordinates to query via alignment
    - Apply diversity transforms (jitter, hinge, wiggle) for different slots
    - Refine with RNA geometry constraints
    - Rank and deduplicate

    Returns list of np.ndarray, each (seq_len, 3).
    """
    if _GLOBAL_ALIGNER is None:
        return []

    tbm_cfg = CONFIG['tbm']
    hits = find_template_hits(
        query_seq, template_index, _GLOBAL_ALIGNER,
        shortlist_size=tbm_cfg['shortlist_size'],
        top_n=tbm_cfg['top_templates'],
    )

    candidates = []
    used = set()

    for hit_idx, hit in enumerate(hits):
        if len(candidates) >= N_PREDICTIONS:
            break
        if hit['similarity'] < tbm_cfg['min_similarity']:
            break
        if hit['percent_identity'] < tbm_cfg['min_percent_identity']:
            break
        if hit['target_id'] in used:
            continue

        rng = np.random.default_rng((row_idx * 1000003 + hit_idx * 10007 + seed) % (2**32))
        adapted = adapt_template_to_query(query_seq, hit['sequence'], hit['coords'])

        # Apply different diversity transforms for each slot
        slot = len(candidates)
        if slot == 0:
            transformed = adapted
        elif slot == 1:
            noise_scale = max(0.01, (0.40 - hit['similarity']) * 0.06)
            transformed = adapted + rng.normal(0, noise_scale, adapted.shape).astype(np.float32)
        elif slot == 2:
            longest_seg = max(segments, key=lambda s: s[1] - s[0])
            transformed = apply_hinge(adapted, longest_seg, rng)
        elif slot == 3:
            transformed = jitter_chains(adapted, segments, rng)
        else:
            transformed = smooth_wiggle(adapted, segments, rng)

        refined = adaptive_rna_constraints(transformed, segments, confidence=hit['similarity'])
        candidates.append(refined)
        used.add(hit['target_id'])

    return rank_candidates(candidates, segments, N_PREDICTIONS)


print(f'TBM pipeline defined. Functions: load_training_coords, TemplateIndex, '
      f'find_template_hits, adapt_template_to_query, adaptive_rna_constraints, '
      f'tbm_predict, generate_rna_helix, rank_candidates')

In [ ]:
@torch.no_grad()
def predict_structure(model, sequence, max_seq_len, device):
    """Predict 5 3D structures for an RNA sequence using our trained model."""
    tokens = encode_sequence(sequence)
    seq_len = len(tokens)

    if seq_len <= max_seq_len:
        padded = tokens + [0] * (max_seq_len - seq_len)
        mask = [1] * seq_len + [0] * (max_seq_len - seq_len)
        tokens_t = torch.tensor([padded], dtype=torch.long, device=device)
        mask_t = torch.tensor([mask], dtype=torch.bool, device=device)
        pred = model(tokens_t, mask_t)
        coords = pred['coords'][0, :seq_len].cpu().numpy()
        confidence = pred['confidence'][0, :seq_len].cpu().numpy()
    else:
        # Sliding window with overlap for long sequences
        window = max_seq_len
        stride = max_seq_len // 2
        coords = np.zeros((seq_len, model.num_predictions, 3), dtype=np.float32)
        confidence = np.zeros((seq_len, model.num_predictions), dtype=np.float32)
        weights = np.zeros((seq_len, 1, 1), dtype=np.float32)
        for start in range(0, seq_len, stride):
            end = min(start + window, seq_len)
            chunk = tokens[start:end]
            chunk_len = len(chunk)
            padded = chunk + [0] * (window - chunk_len)
            mask = [1] * chunk_len + [0] * (window - chunk_len)
            tokens_t = torch.tensor([padded], dtype=torch.long, device=device)
            mask_t = torch.tensor([mask], dtype=torch.bool, device=device)
            pred = model(tokens_t, mask_t)
            coords[start:end] += pred['coords'][0, :chunk_len].cpu().numpy()
            confidence[start:end] += pred['confidence'][0, :chunk_len].cpu().numpy()
            weights[start:end] += 1.0
            if end >= seq_len:
                break
        coords = coords / np.maximum(weights, 1e-8)
        confidence = confidence / np.maximum(weights[:, :, 0], 1e-8)

    return {'coords': coords, 'confidence': confidence}


# ── Load training data for TBM ──
template_index = None
segments_map = {}

if HAS_BIOPYTHON:
    train_lbl_path = CONFIG['data']['train_labels_csv']
    val_lbl_path = CONFIG['data']['val_labels_csv']
    train_seq_path = CONFIG['data']['train_sequences_csv']
    val_seq_path = CONFIG['data']['val_sequences_csv']

    if os.path.exists(train_seq_path) and os.path.exists(train_lbl_path):
        print('Loading training data for TBM...')
        t0 = time.time()

        train_seqs_df = pd.read_csv(train_seq_path)
        if os.path.exists(val_seq_path):
            val_seqs_df = pd.read_csv(val_seq_path)
            train_seqs_df = pd.concat([train_seqs_df, val_seqs_df], ignore_index=True)

        train_coords_dict = load_training_coords(train_lbl_path)
        if os.path.exists(val_lbl_path):
            val_coords = load_training_coords(val_lbl_path)
            train_coords_dict.update(val_coords)

        # Build fast template index with k-mer pre-filtering
        template_index = TemplateIndex(train_seqs_df, train_coords_dict, k=3)

        # Build chain segment map from test data
        segments_map = build_segments_map(test_df)

        print(f'  Template pool: {len(template_index.entries)} structures '
              f'({time.time()-t0:.1f}s)')
    else:
        print('Training data not found. TBM disabled, using model-only predictions.')

# ── Generate predictions: TBM first, model fallback, helix last resort ──
rows = []
max_seq_len = CONFIG['data']['max_seq_len']
tbm_count = 0
model_fallback_count = 0
helix_fallback_count = 0

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting'):
    target_id = row['target_id']
    sequence = row['sequence']
    seq_len = len(sequence)
    segments = segments_map.get(target_id, [(0, seq_len)])

    combined = []

    # Phase 1: TBM predictions (primary — uses real template structures)
    if HAS_BIOPYTHON and template_index is not None:
        tbm_preds = tbm_predict(
            sequence, template_index, segments,
            row_idx=idx, seed=CONFIG['training']['seed'],
        )
        combined.extend(tbm_preds)
        if len(tbm_preds) > 0:
            tbm_count += 1

    # Phase 2: Fill remaining slots with trained model predictions
    n_tbm = len(combined)
    if len(combined) < N_PREDICTIONS:
        result = predict_structure(model, sequence, max_seq_len, device)
        model_coords = result['coords']  # (seq_len, num_preds, 3)
        for p in range(min(model_coords.shape[1], N_PREDICTIONS - len(combined))):
            pred_coords = model_coords[:, p, :]
            # Apply RNA constraints to model predictions too
            refined = adaptive_rna_constraints(pred_coords, segments, confidence=0.3)
            combined.append(refined)
        if n_tbm == 0:
            model_fallback_count += 1

    # Phase 3: Fill any remaining with idealized helix (last resort)
    while len(combined) < N_PREDICTIONS:
        seed_val = idx * 1000000 + len(combined) * 1000
        dn = generate_rna_helix(sequence, seed=seed_val)
        combined.append(adaptive_rna_constraints(dn, segments, confidence=0.2))
        helix_fallback_count += 1

    # Rank all candidates by geometry quality
    combined = rank_candidates(combined, segments, N_PREDICTIONS)

    # If rank_candidates filtered too many, pad with helix
    while len(combined) < N_PREDICTIONS:
        seed_val = idx * 1000000 + len(combined) * 7777
        dn = generate_rna_helix(sequence, seed=seed_val)
        combined.append(adaptive_rna_constraints(dn, segments, confidence=0.2))

    # Write rows for submission
    for resid, nuc in enumerate(sequence):
        entry = {
            'ID': f'{target_id}_{resid + 1}',
            'resname': nuc.upper(),
            'resid': resid + 1,
        }
        for p in range(N_PREDICTIONS):
            if resid < len(combined[p]):
                x, y, z = combined[p][resid]
            else:
                x, y, z = 0.0, 0.0, 0.0
            # Clip to competition coordinate range
            x = max(-999.999, min(9999.999, float(x)))
            y = max(-999.999, min(9999.999, float(y)))
            z = max(-999.999, min(9999.999, float(z)))
            entry[f'x_{p+1}'] = round(x, 3)
            entry[f'y_{p+1}'] = round(y, 3)
            entry[f'z_{p+1}'] = round(z, 3)
        rows.append(entry)

submission = pd.DataFrame(rows)
print(f'\nPrediction summary:')
print(f'  Targets with TBM predictions: {tbm_count}')
print(f'  Targets model-only fallback: {model_fallback_count}')
print(f'  Helix fallback slots used: {helix_fallback_count}')
print(f'  N_PREDICTIONS per target: {N_PREDICTIONS}')
print(f'\nSubmission shape: {submission.shape}')
print(submission.head(10))

## 9. Save Submission

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(output_path, index=False)
print(f'Submission saved to {output_path}')
print(f'Total rows: {len(submission)}')
print(f'Unique targets: {submission["ID"].str.rsplit("_", n=1).str[0].nunique()}')
print(f'Predictions per target: {N_PREDICTIONS}')

# Sanity checks
coord_cols = [c for c in submission.columns if c.startswith(('x_', 'y_', 'z_'))]
print(f'\nCoordinate columns: {len(coord_cols)} ({N_PREDICTIONS} predictions x 3 axes)')
print(f'\nCoordinate statistics:')
print(submission[coord_cols].describe())